# Post-processing: 3-lvl polish results

Loads the polished trajectory and pre-computed ε-sweep data from the overnight `robust_iswap_polish_3lvl.jl` run. Plots:

- **Fidelity F_3 vs ε (linear)** — robust vs default, per error channel (n̂_1, n̂_2, n̂_1·n̂_2)
- **Infidelity 1 − F_3 vs ε (log)** — same

All curves are RAW — no virtual Z. Default = calibrated π/4 Gaussian-square, no MW. Robust = polished 3-lvl gate.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)

using JLD2, CairoMakie, Printf, LinearAlgebra

## Load polished trajectory and ε-sweep data

In [ ]:
const RUN_TAG    = "polish_3lvl_robust_2lvl_4dim_geff2MHz_200ns_eta170"
const TRAJ_PATH  = joinpath(@__DIR__, "traj_$(RUN_TAG)_profile.jld2")
const SWEEP_PATH = joinpath(@__DIR__, "eps_sweep_$(RUN_TAG).jld2")

@load TRAJ_PATH traj_polish U_goal U_iSWAP V_rise V_fall A_rise A_fall
@load SWEEP_PATH εs_MHz_2q F_rob F_def L_rob L_def

@printf("Loaded trajectory: %s\n", TRAJ_PATH)
@printf("  knots          : %d\n", size(traj_polish[:u], 2))
@printf("  MW duration    : %.4f ns\n", traj_polish[:t][end])
@printf("  Δt             : %.4f ns\n", traj_polish[:t][2] - traj_polish[:t][1])
@printf("\nLoaded ε-sweep:   %s\n", SWEEP_PATH)
@printf("  ε grid points  : %d   range [%.1f, %.1f] MHz\n",
    length(εs_MHz_2q), minimum(εs_MHz_2q), maximum(εs_MHz_2q))
@printf("  channels       : %s\n", join(string.(keys(F_rob)), ", "))

i0 = argmin(abs.(εs_MHz_2q))
@printf("\nAt ε = 0:\n")
for ch in keys(F_rob)
    @printf("  %-6s  robust F=%.6f  L=%.3e    default F=%.6f  L=%.3e\n",
        string(ch), F_rob[ch][i0], L_rob[ch][i0], F_def[ch][i0], L_def[ch][i0])
end

## Fidelity F_3 vs ε  —  linear scale, 3 panels

In [ ]:
channel_keys   = collect(keys(F_rob))
channel_titles = Dict(:n̂1 => "n̂_1  (qubit-1 dephasing)",
                      :n̂2 => "n̂_2  (qubit-2 dephasing)",
                      :n̂1n̂2 => "n̂_1·n̂_2  (cross-Kerr / ZZ)")
channel_colors = Dict(:n̂1 => :crimson, :n̂2 => :forestgreen, :n̂1n̂2 => :royalblue)

fig_F = Figure(size = (1500, 500), fontsize = 16)
for (i, ch) in enumerate(channel_keys)
    ax = Axis(fig_F[1, i]; xlabel = "ε (MHz)", ylabel = "F_3",
        title = get(channel_titles, ch, string(ch)))
    c = get(channel_colors, ch, :black)
    lines!(ax, εs_MHz_2q, F_rob[ch]; color = c, linewidth = 2.5, label = "robust")
    lines!(ax, εs_MHz_2q, F_def[ch]; color = c, linewidth = 1.5, linestyle = :dash, label = "default")
    ylims!(ax, 0.0, 1.02)
    axislegend(ax; position = :lb, labelsize = 11)
end
save(joinpath(@__DIR__, "postprocess_fidelity_linear_$(RUN_TAG).png"), fig_F; px_per_unit = 4)
display(fig_F)
fig_F

## Infidelity 1 − F_3 vs ε  —  log scale, 3 panels

In [ ]:
fig_logI = Figure(size = (1500, 500), fontsize = 16)
for (i, ch) in enumerate(channel_keys)
    ax = Axis(fig_logI[1, i]; xlabel = "ε (MHz)", ylabel = "1 − F_3",
        yscale = log10, title = get(channel_titles, ch, string(ch)))
    c = get(channel_colors, ch, :black)
    lines!(ax, εs_MHz_2q, max.(1 .- F_rob[ch], 1e-12); color = c, linewidth = 2.5, label = "robust")
    lines!(ax, εs_MHz_2q, max.(1 .- F_def[ch], 1e-12); color = c, linewidth = 1.5, linestyle = :dash, label = "default")
    axislegend(ax; position = :lb, labelsize = 11)
end
save(joinpath(@__DIR__, "postprocess_infidelity_log_$(RUN_TAG).png"), fig_logI; px_per_unit = 4)
display(fig_logI)
fig_logI

## Leakage L_3 vs ε  —  log scale, 3 panels

In [ ]:
fig_L = Figure(size = (1500, 500), fontsize = 16)
for (i, ch) in enumerate(channel_keys)
    ax = Axis(fig_L[1, i]; xlabel = "ε (MHz)", ylabel = "L_3",
        yscale = log10, title = get(channel_titles, ch, string(ch)))
    c = get(channel_colors, ch, :black)
    lines!(ax, εs_MHz_2q, max.(L_rob[ch], 1e-12); color = c, linewidth = 2.5, label = "robust")
    lines!(ax, εs_MHz_2q, max.(L_def[ch], 1e-12); color = c, linewidth = 1.5, linestyle = :dash, label = "default")
    axislegend(ax; position = :lb, labelsize = 11)
end
save(joinpath(@__DIR__, "postprocess_leakage_log_$(RUN_TAG).png"), fig_L; px_per_unit = 4)
display(fig_L)
fig_L

## Summary table

In [ ]:
@printf("\n%-12s | %-12s | %-12s | %-10s | %-10s\n", "channel", "F robust(0)", "F default(0)", "L rob(0)", "L def(0)")
@printf("%s\n", repeat("-", 78))
for ch in channel_keys
    @printf("%-12s | %.8f | %.8f | %.3e | %.3e\n",
        string(ch), F_rob[ch][i0], F_def[ch][i0], L_rob[ch][i0], L_def[ch][i0])
end

# Crossover ε (where robust starts beating default in infidelity)
@printf("\n%-12s | %-15s\n", "channel", "|ε| where 1−F_rob < 1−F_def")
@printf("%s\n", repeat("-", 50))
for ch in channel_keys
    mask = (1 .- F_rob[ch]) .< (1 .- F_def[ch])
    if any(mask)
        εs_win = εs_MHz_2q[mask]
        @printf("%-12s | robust wins on %.1f%% of ε grid\n",
            string(ch), 100 * count(mask) / length(mask))
    else
        @printf("%-12s | (default always wins on this grid)\n", string(ch))
    end
end